In [ ]:
import cudf
import cugraph
import cuml
import cupy as cp
import numpy as np
from cuml.metrics import pairwise_distances

# Sample embeddings for two users with different sizes
user1_embeddings = np.random.rand(1000, 4096)  # 100 embeddings of 4096 dimensions
user2_embeddings = np.random.rand(800, 4096)   # 80 embeddings of 4096 dimensions

# Convert to GPU arrays using CuPy
user1_embeddings_gpu = cp.asarray(user1_embeddings)
user2_embeddings_gpu = cp.asarray(user2_embeddings)

# Compute the pairwise cosine similarity matrix
cost_matrix = pairwise_distances(user1_embeddings_gpu, user2_embeddings_gpu, metric='cosine')

# Convert the cost matrix to a cuDF DataFrame for CuGraph
rows, cols = cost_matrix.shape
df = cudf.DataFrame({'weight': cost_matrix.ravel(order='C')})  # 'C' denotes row-major order
costs_series = df['weight']

In [ ]:
%%time
# Use the dense Hungarian algorithm with non-square matrix support
cost, assignment = cugraph.dense_hungarian(costs_series, rows, cols)

# Print the results
print("Total cost of matching:", cost)
print("Assignment of embeddings:")
print(assignment)

- 1k embeddings: 2s exec time, 30% a100 utilization
- 10k embeddings: 1m exec time, 80% a100 utilization